In [ ]:
import json
json_path = "Datasets/test_1_w_spec.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)
# =========================
# Collect unique topo pairs
# =========================
topo_counter = Counter()

for item in data:
    spec = item.get("extended_spec", {})
    topo = spec.get("topo", {})

    t = topo.get("type", "")
    sub = topo.get("sub", "")

    # Normalize type → list
    if isinstance(t, list):
        types = [str(x).lower() for x in t]
    elif isinstance(t, str):
        types = [t.lower()]
    else:
        types = ["unknown"]

    sub = sub.lower() if isinstance(sub, str) else ""

    for tt in types:
        topo_counter[(tt, sub)] += 1

# =========================
# Print results
# =========================
print("Unique (type, sub) pairs:")
print("-" * 50)

for (t, s), cnt in sorted(topo_counter.items(), key=lambda x: -x[1]):
    print(f"type: {t:<15} | sub: {s:<15} | count: {cnt}")

print("-" * 50)
print(f"Total unique topo pairs: {len(unique_topo)}")

In [2]:
import json
import pandas as pd
from collections import Counter

# =========================
# Load Data
# =========================
csv_path = "Results/qwen_dual_lora_results_test_2_markdown.csv"
json_path = "Datasets/test_2_w_spec.json"

df = pd.read_csv(csv_path)

# --- FILTER: Only Test 1 ---
df = df[df["Split"] == "Test 2"].copy()
print(len(df))


# Load JSON spec
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

spec_map = {
    item["claim"]: item["extended_spec"]
    for item in data
}

# =========================
# Your TOPO MAP (with restriction)
# =========================
_TOPO_MAP = {
    ("bar",  "vertical"):   "barchart_vertical",
    ("bar",  "horizontal"): "barchart_horizontal",
    ("bar",  "stacked"):    "barchart_vertical",
    ("bar",  ""):           "barchart_vertical",

    ("line", ""):           "line_chart",
    ("line", "area"):       "line_chart",

    ("pie",  ""):           "pie_chart",
    ("pie",  "donut"):      "pie_chart",

    ("scatter", ""):        "scatter_plot",

    # These will be collapsed later (not in your allowed set)
    ("bubble",  ""):        "bubble_chart",
    ("map",     ""):        "map_chart",
}

# Allowed final label space
_ALLOWED = {
    "pie_chart",
    "line_chart",
    "barchart_vertical",
    "barchart_horizontal",
    "histogram",
    "unknown",
    "bar+line",
    "area",
    "scatter_plot",
}

# =========================
# Mapping Function (robust)
# =========================
def map_spec_to_fixed_label(spec, csv_label):
    topo = spec.get("topo", {})
    
    t = topo.get("type", "")
    sub = topo.get("sub", "")

    # Normalize
    if isinstance(t, list):
        types = [str(x).lower() for x in t]
    elif isinstance(t, str):
        types = [t.lower()]
    else:
        types = []

    sub = sub.lower() if isinstance(sub, str) else ""

    # =========================
    # HARD FALLBACK (your rule)
    # =========================
    if (not types or types == [""]) and sub == "":
        return csv_label

    # =========================
    # COMBO CASES
    # =========================
    if "bar+line" in types or ("bar" in types and "line" in types):
        return "bar+line"

    if "area+line" in types:
        return "area"

    # =========================
    # PIE FAMILY
    # =========================
    if "pie" in types:
        return "pie_chart"

    if "circle" in types and sub == "donut":
        return "pie_chart"

    # =========================
    # BAR
    # =========================
    if "bar" in types:
        if sub == "horizontal":
            return "barchart_horizontal"
        return "barchart_vertical"

    # =========================
    # LINE
    # =========================
    if "line" in types:
        return "line_chart"

    # =========================
    # HISTOGRAM
    # =========================
    if "histogram" in types:
        return "histogram"

    # =========================
    # AREA
    # =========================
    if "area" in types:
        return "area"

    # =========================
    # SCATTER
    # =========================
    if "scatter" in types:
        return "scatter_plot"

    # =========================
    # FINAL FALLBACK
    # =========================
    return csv_label


def final_chart_type(row):
    spec = spec_map.get(row["Claim"], None)

    if spec is None:
        return row["Chart Type"]
    

    return map_spec_to_fixed_label(spec, row["Chart Type"])


# =========================
# Apply Mapping
# =========================
df["Mapped Chart Type"] = df.apply(final_chart_type, axis=1)

# =========================
# Fine-grain print function (RETAINED)
# =========================
def print_fine_grain_accuracy(df, chart_col="Mapped Chart Type"):
    print("\nPer-Chart-Type Accuracy")
    print("-" * 60)

    results = []

    for chart_type, group in df.groupby(chart_col):
        total = len(group)
        correct = (group["Predicted Verdict"] == group["True Verdict"]).sum()
        acc = (correct / total) * 100 if total > 0 else 0

        results.append((chart_type, acc, total))

    # sort by count (like your previous output)
    results = sorted(results, key=lambda x: -x[2])

    for chart_type, acc, total in results:
        print(f"{chart_type:<30} {acc:>6.1f}%    {total}")

    print("-" * 60)
    print(f"Total samples accounted for: {len(df)} / {len(df)}")


print("\n--- Original Labels ---")
print_fine_grain_accuracy(df, chart_col="Chart Type")

print("\n--- Mapped Labels ---")
print_fine_grain_accuracy(df, chart_col="Mapped Chart Type")

# =========================
# Debug: changed rows
# =========================
changed = df[df["Chart Type"] != df["Mapped Chart Type"]]

print(f"\nChanged rows: {len(changed)} / {len(df)}")
display(changed.head(10))

981

--- Original Labels ---

Per-Chart-Type Accuracy
------------------------------------------------------------
pie_chart                        81.5%    303
unknown                          77.7%    238
barchart_vertical                78.0%    205
line_chart                       72.8%    169
barchart_horizontal              84.8%    66
------------------------------------------------------------
Total samples accounted for: 981 / 981

--- Mapped Labels ---

Per-Chart-Type Accuracy
------------------------------------------------------------
pie_chart                        81.1%    433
barchart_vertical                79.3%    241
line_chart                       72.0%    189
barchart_horizontal              84.1%    69
histogram                        47.8%    23
area                             90.0%    20
scatter_plot                    100.0%    6
------------------------------------------------------------
Total samples accounted for: 981 / 981

Changed rows: 303 / 981


,Split,Chart Type,Reasoning Type,Claim,True Verdict,Predicted Verdict,Verdict Match,True Rationale,Predicted Rationale,Mapped Chart Type
22,Test 2,unknown,unknown,"In 1980, the population was 27861.",SUPPORTED,SUPPORTED,True,The red dot for the year 1980 shows that the p...,The red line indicates the population for a gi...,line_chart
23,Test 2,unknown,unknown,The difference between population of 1910 and ...,REFUTED,REFUTED,True,As the chart shows the difference between 1880...,"Based on the graph for Watertown, NY populatio...",line_chart
24,Test 2,unknown,unknown,"According to the chart, in the year 1950 there...",SUPPORTED,SUPPORTED,True,The highest population value recorded in the c...,"In the line graph, the total population value ...",line_chart
25,Test 2,unknown,unknown,"Watertown, NY reached its peak population in 1...",SUPPORTED,REFUTED,False,The highest population point on the line is se...,The chart shows that Watertown's population co...,line_chart
26,Test 2,unknown,unknown,The population at the end of the graph has inc...,SUPPORTED,REFUTED,False,The chart starts with 9336 population and fini...,The line on the graph is decreasing from left ...,line_chart
27,Test 2,unknown,unknown,The largest population increase is related to ...,REFUTED,SUPPORTED,False,"As the chart shows, the largest population inc...","The population increased by 12,000 people betw...",line_chart
28,Test 2,unknown,unknown,The population is constantly increasing with t...,REFUTED,SUPPORTED,False,"As the chart shows, the population decreased b...",In the year 1900 the population was 20000 and ...,line_chart
29,Test 2,unknown,unknown,The population of Watertown in 2020 had the lo...,REFUTED,SUPPORTED,False,1870 was the year with the lowest population o...,The red line representing Watertown population...,line_chart
36,Test 2,unknown,unknown,"According to the graph, we can see decreasing ...",SUPPORTED,SUPPORTED,True,"According to the graph, we can see the decreas...",The line is decreasing from 9 to 25 years and ...,barchart_vertical
37,Test 2,unknown,unknown,Blue colour bars are used in the given chart.,REFUTED,REFUTED,True,Only red color bars is used to represent data ...,Red colour bars are used in the given chart.,barchart_vertical
